# PerFedAvg Server

> AS implemented in https://proceedings.neurips.cc/paper/2020/file/24389bfe4fe2eba8bf9aa9203a44cdad-Paper.pdf

In [ ]:
#| default_exp servers.perfedavg

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fastcore.utils import *
from fastcore.all import *

import torch
import torch.nn.functional as F

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

## Per-FedAvg

Personalized Federated Learning with Theoretical Guarantees: A Model-Agnostic Meta-Learning Approach

In [ ]:
#| export
@AlgorithmRegistry.register_server("perfedavg")
class ServerPerFedAvg(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 

### Evaluation

In [ ]:
#| export
@patch
def evaluate(self: ServerPerFedAvg, t):
    self.cfg.agg ="mtl"
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):

        client_state = self.state_mgr.get_state(id)
        client = self.client_fn(client_cls= self.client_cls, id= id, cfg= self.cfg, comm_round= t, client_state= client_state)
        client.train_one_step()
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
    self.cfg.agg = "one_model"
    return lst_train_res, lst_test_res    


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()